In [35]:
import requests

def get_nws_weather(latitude: float, longitude: float) -> dict:
  """
  Retrieves weather data from the National Weather Service API for a given latitude and longitude.
  """

  base_url = "https://api.weather.gov"
  user_agent = "MyWeatherApp (tmontgomery)"
  headers = {"User-Agent": user_agent,"Accept": "application/json"}
  weather_data = {}

  try:
    # Get gridpoint metadata
    points_url = f"{base_url}/points/{latitude},{longitude}"
    points_response = requests.get(points_url, headers=headers)
    points_response.raise_for_status()
    points_data = points_response.json()

    # Get city name and state
    city = points_data.get('properties', {}).get('relativeLocation', {}).get('properties', {}).get('city')
    state = points_data.get('properties', {}).get('relativeLocation', {}).get('properties', {}).get('state')

    # Store basic location info
    weather_data["coordinates"] = {"Latitude": latitude, "Longitude": longitude}
    weather_data["city"] = city if city else "Unknown City"
    weather_data["state"] = state if state else "Unknown State"

    forecast_url = points_data.get('properties', {}).get('forecast')

    # Get the detailed forecast
    forecast_response = requests.get(forecast_url, headers=headers)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()

    # Extract first forecast period
    periods = forecast_data.get('properties', {}).get('periods', [])
    current_period = periods[0]
    temperature = f"{current_period.get('temperature')} {current_period.get('temperatureUnit')}"
    detailed_forecast = current_period.get('detailedForecast')

    weather_data["temperature"] = temperature if temperature else "N/A"
    weather_data["detailed_forecast"] = detailed_forecast if detailed_forecast else "No detailed forecast available."

    return weather_data

  except requests.exceptions.RequestException as e:
    print(f"Error fetching weather data: {e}")
    return {}

  except KeyError as e:
    print(f"Error parsing NWS API response: Missing key {e}")
    return {}

  return weather_data

#Enter lat and lon to display weather data
if __name__ == "__main__":
    # Example: Elizabethtown, NC
    lat = 34.6272
    lon = -78.6107

    weather_info = get_nws_weather(lat, lon)

    if weather_info:
        print("\n--- Weather Information ---")
        print(f"Coordinates: {weather_info.get('coordinates')}")
        print(f"City: {weather_info.get('city')}, {weather_info.get('state')}")
        print(f"Temperature: {weather_info.get('temperature')}")
        print(f"Detailed Forecast: {weather_info.get('detailed_forecast')}")
    else:
        print("\nFailed to retrieve weather information.")

print("\n--- Example: Raleigh, NC ---")
lat_rdu = 35.7861
lon_rdu = -78.6635
weather_info_rdu = get_nws_weather(lat_rdu, lon_rdu)
if weather_info_rdu:
  print("\n--- Weather Information ---")
  print(f"Coordinates: {weather_info_rdu.get('coordinates')}")
  print(f"City, State: {weather_info_rdu.get('city')}, {weather_info_rdu.get('state')}")
  print(f"Temperature: {weather_info_rdu.get('temperature')}")
  print(f"Detailed Forecast: {weather_info_rdu.get('detailed_forecast')}")

else:
  print("\nFailed to retrieve weather information for Raleigh, NC.")

print("\n--- Example: Biloxi, MS ---")
lat_gpt = 30.4097
lon_gpt = -88.9262
weather_info_gpt = get_nws_weather(lat_gpt, lon_gpt)
if weather_info_gpt:
  print("\n--- Weather Information ---")
  print(f"Coordinates: {weather_info_gpt.get('coordinates')}")
  print(f"City, State: {weather_info_gpt.get('city')}, {weather_info_gpt.get('state')}")
  print(f"Temperature: {weather_info_gpt.get('temperature')}")
  print(f"Detailed Forecast: {weather_info_gpt.get('detailed_forecast')}")

else:
  print("\nFailed to retrieve weather information for Biloxi, MS.")


--- Weather Information ---
Coordinates: {'Latitude': 34.6272, 'Longitude': -78.6107}
City: Elizabethtown, NC
Temperature: 54 F
Detailed Forecast: Mostly cloudy, with a high near 54. Northeast wind around 6 mph.

--- Example: Raleigh, NC ---

--- Weather Information ---
Coordinates: {'Latitude': 35.7861, 'Longitude': -78.6635}
City, State: Raleigh, NC
Temperature: 49 F
Detailed Forecast: A chance of rain. Cloudy, with a high near 49. Northeast wind around 8 mph. Chance of precipitation is 40%.

--- Example: Biloxi, MS ---

--- Weather Information ---
Coordinates: {'Latitude': 30.4097, 'Longitude': -88.9262}
City, State: Biloxi, MS
Temperature: 73 F
Detailed Forecast: Sunny. High near 73, with temperatures falling to around 70 in the afternoon. Southeast wind around 10 mph.
